
This Jupyter Notebook is part of a **collection of scripts curated by Tarlan Ahadli** for the **2025 Spring Semester**, based on **projects provided by Prof. Hajder Levente**. These scripts cover a range of computational topics, including **computer vision, image processing, numerical computing, and matrix transformations**, serving as educational resources for academic use.

### **Usage Restrictions**
This material is **strictly for academic and educational purposes**.  
**Commercial use is prohibited.**

For any **issues, suggestions, or inquiries**, please contact:  
📧 **tarlanahad@gmail.com**

In [2]:

# === 1) DOWNLOAD & UNZIP THE DATASET ===
!wget -q https://vision.middlebury.edu/stereo/data/scenes2005/FullSize/zip-2views/Art-2views.zip
!unzip -o Art-2views.zip  # -o to overwrite if re-running


Archive:  Art-2views.zip
  inflating: Art/disp1.png           
  inflating: Art/disp5.png           
 extracting: Art/dmin.txt            
  inflating: Art/view1.png           
  inflating: Art/view5.png           


In [4]:
# Calculate Disparity Map
# Disparity to Depth
# Visualize Depth information 3D

In [ ]:
# === 2) IMPORTS ===
import cv2 from google.colab.patches import cv2_imshow  # For showing images in Colab
import numpy as np
from numba import njit
from google.colab.patches import cv2_imshow  # For showing images in Colab


ModuleNotFoundError: No module named 'numpy'

In [ ]:
# === 3) DEFINE BLOCK MATCHING FUNCTIONS ===
@njit
def compute_block_sad(left_block, right_block):
    """
    Sum of Absolute Differences (SAD) between two patches.
    """
    return np.sum(np.abs(left_block - right_block))

@njit
def compute_disparity_loop(img_left, img_right, kernel_size, max_disparity):
    """
    For each pixel in the left image, find best match in right image
    by searching up to 'max_disparity' pixels to the left.
    """
    height, width = img_left.shape
    disparity_map = np.zeros((height, width), dtype=np.uint8)
    offset = kernel_size // 2

    for row in range(offset, height - offset):
        for col in range(offset, width - offset):
            best_score = np.inf
            best_disp = 0

            left_block = img_left[row-offset:row+offset+1, col-offset:col+offset+1]

            for d in range(max_disparity):
                col_r = col - d
                if col_r - offset < 0:
                    break  # Can't go further left; no more valid blocks

                right_block = img_right[row-offset:row+offset+1,
                                        col_r-offset:col_r+offset+1]

                sad = compute_block_sad(left_block, right_block)
                if sad < best_score:
                    best_score = sad
                    best_disp = d

            disparity_map[row, col] = best_disp

    return disparity_map

def compute_disparity_map(view1_path, view5_path, kernel_size=9, max_disparity=100):
    """
    Loads two stereo images, converts to grayscale, and computes a disparity map.
    """
    img_left_color = cv2.imread(view1_path)
    img_right_color = cv2.imread(view5_path)

    # Convert to grayscale
    img_left = cv2.cvtColor(img_left_color, cv2.COLOR_BGR2GRAY)
    img_right = cv2.cvtColor(img_right_color, cv2.COLOR_BGR2GRAY)

    # Numba-accelerated loop for disparity
    disparity_map = compute_disparity_loop(img_left, img_right, kernel_size, max_disparity)
    return disparity_map

In [ ]:
disp = compute_disparity_map("Art/view1.png", "Art/view5.png",
                              kernel_size=3, max_disparity=250)




In [ ]:
#disp[disp<10] = 10000


# Optional post-processing: median blur to reduce noise
disp = cv2.medianBlur(disp, 3)

# Show the raw disparity result in 2D
cv2_imshow(disp)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [ ]:
# === 6) 3D PLOTTING WITH PLOTLY ===
import plotly.graph_objects as go
# === 5) DISPARITY -> DEPTH CONVERSION ===
# Middlebury 2005 FullSize: focal length f=3740 , baseline B=160 mm
f = 3740.0
B = 160

def disparity_to_depth(disp_map, focal_length=f, baseline=B):
    """
    Converts a disparity map (uint8) to a depth map in millimeters.
    Z = f * B / d
    """
    height, width = disp_map.shape
    depth_map = np.zeros((height, width), dtype=np.float32)

    for y in range(height):
        for x in range(width):
            d = float(disp_map[y, x])
            if d > 0:  # avoid division by zero
                Z = (focal_length * baseline) / (d)  # in mm
                depth_map[y, x] = Z
            else:
                depth_map[y, x] = 0.0  # or np.nan for invalid

    return depth_map

# Compute depth
depth_map = disparity_to_depth(disp)

def plot_depth_map_3d(depth_map, img_color, max_depth=10000, step=5):
    """
    Plots a depth map as a 3D scatter with original image colors.
    """
    # Downsample depth map
    depth_map = depth_map[::step, ::step]
    # Downsample color image to match depth map resolution
    img_color_down = img_color[::step, ::step, :]

    H, W = depth_map.shape
    X, Y = np.meshgrid(np.arange(W), np.arange(H))
    x_coords = X.flatten()
    y_coords = Y.flatten()
    z_coords = depth_map.flatten()

    # Flatten color image into RGB values
    colors_flat = img_color_down.reshape(-1, 3)  # Shape: (H*W, 3)

    # Filter invalid depths (e.g., NaN or 0)
    valid_mask = ~np.isnan(z_coords)
    x_coords = x_coords[valid_mask]
    y_coords = y_coords[valid_mask]
    z_coords = z_coords[valid_mask]
    colors_valid = colors_flat[valid_mask]

    # Convert RGB values (0-255) to Plotly-compatible 'rgb(r,g,b)' strings
    color_str = [f'rgb({r},{g},{b})' for r, g, b in colors_valid]

    # Create 3D scatter
    trace = go.Scatter3d(
        x=x_coords,
        y=y_coords,
        z=z_coords,
        mode='markers',
        marker=dict(
            size=2,
            color=color_str,  # Use original RGB colors
            opacity=0.8
        )
    )

    layout = go.Layout(
        title='3D Depth Map with Original Colors',
        scene=dict(
            xaxis_title='X (downsampled pixel)',
            yaxis_title='Y (downsampled pixel)',
            zaxis_title='Depth (mm)'
        )
    )

    fig = go.Figure(data=[trace], layout=layout)
    fig.show()

# === 7) FINAL 3D VISUALIZATION CALL ===
if __name__ == "__main__":
    # Load the left color image (for RGB colors)
    img_left_color = cv2.imread("Art/view1.png")
    img_left_color_rgb = cv2.cvtColor(img_left_color, cv2.COLOR_BGR2RGB)  # Convert to RGB

    plot_depth_map_3d(depth_map.astype(float), img_left_color_rgb, max_depth=np.inf, step=5)